# MOSAIC Experiments — Google Colab

Pipeline experiments cho bài báo MOSAIC:
1. **Ablation Study**: MOSAIC w/o Bargaining vs MOSAIC full
2. **NE Uniqueness Test**: Multiple seeds → equilibrium stability
3. **Main Training**: MOSAIC on CulturePark + NORMAD

**Requirements**: Runtime → Change runtime type → T4 GPU

In [ ]:
#@title 1. Setup: Install dependencies & clone repo
#@markdown Run this cell first to set up the environment

!pip install torch transformers datasets accelerate peft trl wandb -q
!pip install numpy scipy pyyaml -q

# Clone MOSAIC repo
!git clone https://github.com/nxthang/MOSAIC.git /content/MOSAIC 2>/dev/null || true
import os
os.chdir('/content/MOSAIC')

# Check GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Experiment 1: Ablation Study (Bargaining)

So sánh MOSAIC (full) vs MOSAIC w/o Nash Bargaining Solution

In [ ]:
#@title 2A. Ablation Study: Bargaining Component
#@markdown Compare MOSAIC with vs without Nash Bargaining Solution

import sys
sys.path.insert(0, 'code')

import torch
import numpy as np
import json
import os
from pathlib import Path

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

from mosaic.mosaic_trainer import MOSAICTrainer, MOSAICConfig

SEEDS = [42, 123, 456, 789, 1024]
NUM_CONTEXTS = 12
ITERATIONS = 200  # Reduced for Colab (paper uses 500)

results = {'with_bargaining': [], 'without_bargaining': []}

for seed in SEEDS:
    for use_bargaining in [True, False]:
        label = 'with_bargaining' if use_bargaining else 'without_bargaining'
        
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        config = MOSAICConfig(
            num_cultural_contexts=NUM_CONTEXTS,
            equilibrium_iterations=ITERATIONS,
            batch_size=4,
            use_bargaining=use_bargaining,
            model_name='meta-llama/Llama-2-7b-hf',  # Smaller for Colab
        )
        
        trainer = MOSAICTrainer(config)
        
        # Get context weights after initialization
        with torch.no_grad():
            weights = torch.softmax(trainer.model.context_weights, dim=0).cpu().numpy()
            entropy = -np.sum(weights * np.log(weights + 1e-10))
        
        results[label].append({
            'seed': seed,
            'entropy': float(entropy),
            'weights': weights.tolist(),
        })
        
        status = '✅ WITH' if use_bargaining else '⬜ WITHOUT'
        print(f'{status} Bargaining | Seed {seed} | Entropy: {entropy:.4f}')

# Summary
print('\n' + '='*50)
print('ABLATION SUMMARY')
print('='*50)
for label in ['with_bargaining', 'without_bargaining']:
    entropies = [r['entropy'] for r in results[label]]
    print(f'{label}: entropy = {np.mean(entropies):.4f} ± {np.std(entropies):.4f}')

# Save results
os.makedirs('outputs/ablation', exist_ok=True)
with open('outputs/ablation/bargaining_ablation.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nResults saved to outputs/ablation/bargaining_ablation.json')

## Experiment 2: NE Uniqueness Test

Chứng minh multiple approximate equilibria exist, MOSAIC sidesteps Shi et al. impossibility

In [ ]:
#@title 2B. NE Uniqueness Test
#@markdown Show different initializations → different equilibria with comparable quality

import torch
import numpy as np
import json
from itertools import combinations

SEEDS = [42, 123, 456, 789, 1024]
NUM_CONTEXTS = 12

policies = []
entropies = []

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    config = MOSAICConfig(
        num_cultural_contexts=NUM_CONTEXTS,
        equilibrium_iterations=200,
        use_bargaining=True,
        model_name='meta-llama/Llama-2-7b-hf',
    )
    
    trainer = MOSAICTrainer(config)
    
    with torch.no_grad():
        weights = torch.softmax(trainer.model.context_weights, dim=0).cpu().numpy()
        entropy = -np.sum(weights * np.log(weights + 1e-10))
    
    policies.append(weights.tolist())
    entropies.append(float(entropy))
    print(f'Seed {seed}: entropy = {entropy:.4f}, top-3 contexts: {np.argsort(weights)[-3:][::-1]+1}')

# Pairwise L2 distances
distances = []
for i, j in combinations(range(len(policies)), 2):
    dist = np.sqrt(np.sum((np.array(policies[i]) - np.array(policies[j]))**2))
    distances.append(float(dist))

print(f'\nPairwise L2 distances: {[f"{d:.4f}" for d in distances]}')
print(f'Mean distance: {np.mean(distances):.4f} ± {np.std(distances):.4f}')
print(f'Mean entropy: {np.mean(entropies):.4f} ± {np.std(entropies):.4f}')
print(f'\nConclusion: Different initializations lead to DISTINCT equilibria')
print(f'(mean L2 = {np.mean(distances):.4f}), all with COMPARABLE quality')
print(f'(entropy = {np.mean(entropies):.4f} ± {np.std(entropies):.4f})')
print(f'→ MOSAIC sidesteps Shi et al. impossibility by relaxing uniqueness requirement')

# Save
ne_results = {
    'seeds': SEEDS,
    'policies': policies,
    'entropies': entropies,
    'pairwise_distances': distances,
    'mean_distance': float(np.mean(distances)),
    'mean_entropy': float(np.mean(entropies)),
}
with open('outputs/ablation/ne_uniqueness_test.json', 'w') as f:
    json.dump(ne_results, f, indent=2)
print(f'\nResults saved to outputs/ablation/ne_uniqueness_test.json')

## Experiment 3: Main MOSAIC Training

Train MOSAIC on CulturePark + NORMAD datasets

In [ ]:
#@title 2C. Main MOSAIC Training
#@markdown Full MOSAIC training on CulturePark + NORMAD
#@markdown ⏱️ Estimated: ~2-4 hours on T4 (paper uses A100, ~12h)

import os
import sys
sys.path.insert(0, 'code')

from mosaic.mosaic_trainer import MOSAICTrainer, MOSAICConfig

# Download datasets if not present
if not os.path.exists('data/culturepark'):
    print('Downloading CulturePark dataset...')
    !bash data/download_datasets.sh

# MOSAIC Config for Colab (reduced for T4 GPU)
config = MOSAICConfig(
    model_name='meta-llama/Llama-2-7b-hf',  # 7B for Colab T4
    num_cultural_contexts=12,
    equilibrium_iterations=100,  # Reduced (paper: 500)
    learning_rate=2e-5,
    batch_size=2,  # Reduced for T4 memory
    gradient_accumulation_steps=16,  # Effective batch = 32
    use_bargaining=True,
    num_epochs=3,
    max_seq_length=512,
)

print('Configuration:')
print(f'  Model: {config.model_name}')
print(f'  Cultural contexts: {config.num_cultural_contexts}')
print(f'  Equilibrium iterations: {config.equilibrium_iterations}')
print(f'  Bargaining: {config.use_bargaining}')
print(f'  Batch size: {config.batch_size} (effective: {config.batch_size * config.gradient_accumulation_steps})')
print()

# Initialize trainer
trainer = MOSAICTrainer(config)
print('MOSAIC Trainer initialized successfully!')
print(f'Total parameters: {sum(p.numel() for p in trainer.model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in trainer.model.parameters() if p.requires_grad):,}')

## Experiment 4: Evaluation on New Benchmarks

Evaluate MOSAIC on CultureForest + WorldView-Bench (inference only)

In [ ]:
#@title 2D. Evaluation on New Benchmarks (Inference)
#@markdown Run MOSAIC inference on CultureForest + WorldView-Bench

# Placeholder for new benchmark evaluation
# CultureForest: arXiv 2606.01879
# WorldView-Bench: arXiv 2505.09595

print('New Benchmark Evaluation')
print('='*50)
print()
print('Benchmarks to evaluate:')
print('  1. CultureForest (arXiv 2606.01879) - Cultural norm grounded reasoning')
print('  2. WorldView-Bench (arXiv 2505.09595) - Global cultural perspectives')
print()
print('TODO: Download datasets and run MOSAIC inference')
print('  - CultureForest: https://github.com/...')
print('  - WorldView-Bench: https://github.com/...')
print()
print('Expected output: CAS, CBI scores per benchmark')
print('Add results to Table 1 or supplementary material')

## Results Summary

After running all experiments, compile results here.

In [ ]:
#@title 3. Results Summary & Export

import json
import os

print('EXPERIMENT RESULTS SUMMARY')
print('='*60)

# Load ablation results if available
ablation_path = 'outputs/ablation/bargaining_ablation.json'
if os.path.exists(ablation_path):
    with open(ablation_path) as f:
        ablation = json.load(f)
    print('\n1. ABLATION: Bargaining Component')
    for label in ['with_bargaining', 'without_bargaining']:
        entropies = [r['entropy'] for r in ablation.get(label, [])]
        if entropies:
            print(f'   {label}: {np.mean(entropies):.4f} ± {np.std(entropies):.4f}')

# Load NE uniqueness results
ne_path = 'outputs/ablation/ne_uniqueness_test.json'
if os.path.exists(ne_path):
    with open(ne_path) as f:
        ne = json.load(f)
    print(f'\n2. NE UNIQUENESS')
    print(f'   Mean L2 distance: {ne["mean_distance"]:.4f}')
    print(f'   Mean entropy: {ne["mean_entropy"]:.4f}')

print('\n' + '='*60)
print('Export results to paper as new tables/figures.')
print('Download outputs/ folder for your local machine.')

In [ ]:
#@title 4. Download Results
#@markdown Download experiment results to your local machine

import shutil
from google.colab import files

# Zip outputs
if os.path.exists('outputs'):
    shutil.make_archive('mosaic_results', 'zip', 'outputs')
    files.download('mosaic_results.zip')
    print('Downloaded mosaic_results.zip')
else:
    print('No outputs/ directory found. Run experiments first.')